In [1]:
import random
import math
import re
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
import pandas as pd

# ----------------------------
# 1. HÀM XỬ LÝ TRẠNG THÁI
# ----------------------------

MOVE_ORDER = ["L", "R", "U", "D"]  # L/R/U/D là hướng đi của ô trống 0

def parse_state(text):
    """
    Nhập được nhiều dạng:
    1 2 3/4 5 6/7 8 0
    1 2 3
    4 5 6
    7 8 0
    """
    nums = list(map(int, re.findall(r"-?\d+", text)))
    if len(nums) != 9:
        raise ValueError("Trạng thái phải có đúng 9 số, gồm 0 đến 8.")
    if sorted(nums) != list(range(9)):
        raise ValueError("Trạng thái phải chứa đủ các số 0,1,2,3,4,5,6,7,8 và không lặp.")
    return tuple(nums)

def state_to_matrix(state):
    return [list(state[i:i+3]) for i in range(0, 9, 3)]

def state_to_text(state):
    m = state_to_matrix(state)
    return "\n".join(" ".join(str(x) for x in row) for row in m)

def short_state(state):
    return "/".join(" ".join(str(x) for x in state[i:i+3]) for i in range(0, 9, 3))

def goal_positions(goal):
    return {value: (idx // 3, idx % 3) for idx, value in enumerate(goal)}

def manhattan(state, goal):
    """
    h(n) = tổng khoảng cách Manhattan của từng ô so với Goal.
    Không tính ô trống 0.
    """
    pos = goal_positions(goal)
    total = 0
    for idx, value in enumerate(state):
        if value == 0:
            continue
        r, c = idx // 3, idx % 3
        gr, gc = pos[value]
        total += abs(r - gr) + abs(c - gc)
    return total

def misplaced_tiles(state, goal):
    """
    h(n) = số ô sai vị trí, không tính ô trống 0.
    """
    return sum(1 for a, b in zip(state, goal) if a != 0 and a != b)

def inversion_parity_relative_to_goal(state, goal):
    """
    Với 8-puzzle 3x3, trạng thái solvable nếu parity số nghịch thế
    theo thứ tự tương đối của Goal là chẵn.
    """
    order = {tile: i for i, tile in enumerate(goal) if tile != 0}
    seq = [order[tile] for tile in state if tile != 0]
    inv = 0
    for i in range(len(seq)):
        for j in range(i + 1, len(seq)):
            if seq[i] > seq[j]:
                inv += 1
    return inv % 2

def is_solvable(state, goal):
    return inversion_parity_relative_to_goal(state, goal) == 0

def neighbors(state):
    """
    Sinh hàng xóm theo thứ tự L, R, U, D.
    Quy ước: hành động là hướng di chuyển của ô trống 0.
    """
    result = []
    zero = state.index(0)
    r, c = zero // 3, zero % 3

    def swap_and_make(new_r, new_c, action):
        new_zero = new_r * 3 + new_c
        arr = list(state)
        arr[zero], arr[new_zero] = arr[new_zero], arr[zero]
        result.append((action, tuple(arr)))

    if c > 0:
        swap_and_make(r, c - 1, "L")
    if c < 2:
        swap_and_make(r, c + 1, "R")
    if r > 0:
        swap_and_make(r - 1, c, "U")
    if r < 2:
        swap_and_make(r + 1, c, "D")
    return result

def random_solvable_state(goal, min_h=8):
    """
    Tạo trạng thái ngẫu nhiên solvable.
    min_h giúp tránh tạo trạng thái quá gần Goal.
    """
    arr = list(range(9))
    while True:
        random.shuffle(arr)
        st = tuple(arr)
        if is_solvable(st, goal) and manhattan(st, goal) >= min_h:
            return st

# ----------------------------
# 2. HIỂN THỊ ĐẸP TRONG JUPYTER
# ----------------------------

APP_CSS = """
<style>
:root { --puzzle-card:#ffffff; --puzzle-ink:#0f172a; --puzzle-muted:#64748b; --puzzle-line:#e2e8f0; --puzzle-good:#dcfce7; --puzzle-warn:#fef3c7; --puzzle-bad:#fee2e2; }
.puzzle-app { max-width:1120px;margin:0 auto 18px auto;font-family:Inter, Segoe UI, Arial, sans-serif;color:var(--puzzle-ink); }
.puzzle-hero { padding:22px 24px;border-radius:24px;background:linear-gradient(135deg,#1d4ed8 0%,#2563eb 45%,#7c3aed 100%);color:white;box-shadow:0 18px 45px rgba(37,99,235,.22);margin-bottom:16px; }
.puzzle-hero h2 { margin:0;font-size:28px;letter-spacing:.2px; }
.puzzle-hero p { margin:8px 0 0;line-height:1.55;opacity:.92; }
.puzzle-card-title { display:flex;align-items:center;justify-content:space-between;gap:8px;margin-bottom:12px; }
.puzzle-card-title h3 { margin:0;font-size:16px;color:var(--puzzle-ink); }
.puzzle-pill { display:inline-flex;align-items:center;padding:4px 10px;border-radius:999px;background:#eef2ff;color:#3730a3;font-size:12px;font-weight:700; }
.puzzle-section-title { margin:16px 0 8px;font-size:18px;color:#0f172a; }
.puzzle-note { padding:12px 14px;border-radius:16px;background:#f8fafc;border:1px solid #e2e8f0;color:#334155;line-height:1.55; }
.puzzle-alert-good { background:var(--puzzle-good); color:#166534; border:1px solid #86efac; }
.puzzle-alert-warn { background:var(--puzzle-warn); color:#92400e; border:1px solid #fcd34d; }
.puzzle-alert-bad  { background:var(--puzzle-bad);  color:#991b1b; border:1px solid #fecaca; }
.puzzle-boards { display:flex;gap:12px;flex-wrap:wrap;align-items:flex-start; }
.puzzle-board { display:inline-block;padding:13px 14px 14px;border-radius:22px;border:1px solid #e2e8f0;background:white;box-shadow:0 10px 24px rgba(15,23,42,.08);vertical-align:top; }
.puzzle-board-title { font-weight:850;font-size:15px;color:#0f172a;margin-bottom:3px; }
.puzzle-board-sub { min-height:16px;font-size:12px;color:#64748b;margin-bottom:10px;white-space:pre-wrap; }
.puzzle-board-grid { display:grid;grid-template-columns:repeat(3,58px);gap:8px; }
.puzzle-tile { width:58px;height:58px;border-radius:17px;display:flex;align-items:center;justify-content:center;font-size:25px;font-weight:900;border:1px solid rgba(148,163,184,.55);box-shadow:inset 0 -2px 0 rgba(15,23,42,.06), 0 4px 10px rgba(15,23,42,.08); }
.puzzle-table-wrap { max-height:520px;overflow:auto;padding:8px;border:1px solid #e2e8f0;border-radius:16px;background:#ffffff; }
.puzzle-table-wrap table { width:100%;font-size:13px;border-collapse:separate;border-spacing:0;table-layout:auto; }
.puzzle-table-wrap th { background:#eff6ff !important;color:#1e3a8a !important;position:sticky;top:0;z-index:1;font-weight:850;text-align:center; }
.puzzle-table-wrap td, .puzzle-table-wrap th { border:1px solid #e2e8f0 !important;padding:9px 10px !important;vertical-align:middle; }
.puzzle-table-wrap td { white-space:normal !important;line-height:1.45; }
.puzzle-table-wrap td:nth-child(3) { min-width:132px;text-align:center; }
.puzzle-table-wrap td:nth-child(6) { min-width:230px; }
.puzzle-mini-node { display:inline-flex;align-items:center;gap:8px;justify-content:center; }
.puzzle-mini-label { width:28px;height:28px;border-radius:9px;background:#dbeafe;color:#1e3a8a;font-weight:900;display:flex;align-items:center;justify-content:center; }
.puzzle-mini-grid { display:grid;grid-template-columns:repeat(3,25px);gap:3px;padding:5px;border-radius:10px;background:#f8fafc;border:1px solid #cbd5e1; }
.puzzle-mini-cell { width:25px;height:25px;border-radius:6px;background:#e0f2fe;color:#0f172a;font-size:13px;font-weight:800;display:flex;align-items:center;justify-content:center;border:1px solid #dbeafe; }
.puzzle-mini-cell.zero { background:#e2e8f0;color:#94a3b8; }
.algo-switch .widget-toggle-button { min-width:155px; }
</style>
"""

def board_html(state, goal=None, title="", subtitle="", highlight_zero=True):
    cells = []
    for idx, x in enumerate(state):
        if x == 0:
            bg = "linear-gradient(180deg,#f8fafc,#e2e8f0)"
            color = "#94a3b8"
            txt = ""
        elif goal is not None and x == goal[idx]:
            bg = "linear-gradient(180deg,#dcfce7,#bbf7d0)"
            color = "#166534"
            txt = str(x)
        else:
            bg = "linear-gradient(180deg,#dbeafe,#bfdbfe)"
            color = "#1e3a8a"
            txt = str(x)
        cells.append(f"""
            <div class=\"puzzle-tile\" style=\"background:{bg};color:{color};\">{txt}</div>
        """)
    return f"""
    <div class=\"puzzle-board\">
        <div class=\"puzzle-board-title\">{title}</div>
        <div class=\"puzzle-board-sub\">{subtitle}</div>
        <div class=\"puzzle-board-grid\">{''.join(cells)}</div>
    </div>
    """

def show_boards(states, goal, titles=None, subtitles=None, max_boards=20):
    states = states[:max_boards]
    titles = titles or [f"Bước {i}" for i in range(len(states))]
    subtitles = subtitles or [""] * len(states)
    html = '<div class="puzzle-boards">'
    for st, title, sub in zip(states, titles, subtitles):
        html += board_html(st, goal, title, sub)
    html += '</div>'
    display(HTML(html))

def label_name(i):
    """
    0 -> A, 1 -> B, ... 25 -> Z, 26 -> A1 ...
    """
    letters = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    if i < 26:
        return letters[i]
    return letters[i % 26] + str(i // 26)

class Labeler:
    def __init__(self):
        self.state_to_label = {}
        self.counter = 0

    def get(self, state):
        if state not in self.state_to_label:
            self.state_to_label[state] = label_name(self.counter)
            self.counter += 1
        return self.state_to_label[state]

def path_from_parent(parent, goal_state):
    """
    parent[state] = (previous_state, action)
    Trả về list [(state, action_from_previous)]
    """
    path = []
    cur = goal_state
    while cur is not None:
        prev, action = parent.get(cur, (None, None))
        path.append((cur, action))
        cur = prev
    path.reverse()
    return path

def make_log_dataframe(log_rows):
    df = pd.DataFrame(log_rows)
    if len(df) == 0:
        return df
    wanted_cols = [
        "Lần", "Bước", "Node", "h(Node)", "Beam/Current",
        "Frontier / Neighbor đang xét", "Reached", "Chọn", "Ghi chú"
    ]
    cols = [c for c in wanted_cols if c in df.columns]
    return df[cols]

# ----------------------------
# 3. HILL CLIMBING SEARCH
# ----------------------------

def hill_climbing_search(
    start,
    goal,
    heuristic="manhattan",
    max_steps=100,
    allow_sideways=False,
    seed=None
):
    """
    Hill-climbing search cơ bản.

    Ý tưởng:
    - Bắt đầu từ Start.
    - Mỗi bước xét các hàng xóm L/R/U/D.
    - Chọn hàng xóm có h nhỏ nhất.
    - Chỉ đi tiếp nếu hàng xóm tốt hơn hiện tại.
    - Nếu không có hàng xóm tốt hơn thì dừng do kẹt local minimum/plateau.
    """
    if seed is not None:
        random.seed(seed)

    h_func = manhattan if heuristic == "manhattan" else misplaced_tiles
    labeler = Labeler()
    parent = {start: (None, None)}
    reached = {start}
    log_rows = []
    current = start
    labeler.get(current)
    best_seen = current
    best_h = h_func(current, goal)

    for step in range(max_steps + 1):
        cur_h = h_func(current, goal)
        if cur_h < best_h:
            best_seen, best_h = current, cur_h

        if current == goal:
            log_rows.append({
                "Lần": 0,
                "Bước": step,
                "Node": f"{labeler.get(current)}\n{state_to_text(current)}",
                "h(Node)": cur_h,
                "Beam/Current": labeler.get(current),
                "Frontier / Neighbor đang xét": "Đã là Goal",
                "Reached": len(reached),
                "Chọn": "Dừng",
                "Ghi chú": "Tìm thấy nghiệm"
            })
            return {
                "found": True,
                "goal_state": current,
                "parent": parent,
                "log": log_rows,
                "best_state": best_seen,
                "best_h": best_h,
                "labeler": labeler,
                "message": "Đã tìm thấy Goal bằng Hill Climbing Search."
            }

        neigh = []
        for action, nxt in neighbors(current):
            labeler.get(nxt)
            h = h_func(nxt, goal)
            neigh.append((h, action, nxt))

        neigh_sorted = sorted(neigh, key=lambda x: (x[0], MOVE_ORDER.index(x[1])))
        chosen_h, chosen_action, chosen_state = neigh_sorted[0]
        frontier_text = " | ".join(f"({labeler.get(st)},{act},h={h})" for h, act, st in neigh_sorted)
        can_move = chosen_h < cur_h or (allow_sideways and chosen_h == cur_h and chosen_state not in reached)

        if can_move:
            parent.setdefault(chosen_state, (current, chosen_action))
            reached.add(chosen_state)
            choose_text = f"{labeler.get(chosen_state)} qua {chosen_action}, h={chosen_h}"
            note = "Đi sang hàng xóm tốt hơn" if chosen_h < cur_h else "Đi ngang plateau"
        else:
            choose_text = "Không chọn"
            note = "Kẹt local minimum/plateau/shoulder → dừng"

        log_rows.append({
            "Lần": 0,
            "Bước": step,
            "Node": f"{labeler.get(current)}\n{state_to_text(current)}",
            "h(Node)": cur_h,
            "Beam/Current": labeler.get(current),
            "Frontier / Neighbor đang xét": frontier_text,
            "Reached": len(reached),
            "Chọn": choose_text,
            "Ghi chú": note
        })

        if can_move:
            current = chosen_state
        else:
            break

    return {
        "found": False,
        "goal_state": None,
        "parent": parent,
        "log": log_rows,
        "best_state": best_seen,
        "best_h": best_h,
        "labeler": labeler,
        "message": f"Chưa tìm thấy Goal. Hill Climbing bị kẹt, trạng thái tốt nhất có h={best_h}."
    }

# ----------------------------
# 3. RANDOM-RESTART HILL CLIMBING
# ----------------------------

def random_restart_hill_climbing(
    start,
    goal,
    heuristic="manhattan",
    max_restarts=20,
    max_steps_per_restart=50,
    use_given_start_first=True,
    allow_sideways=False,
    seed=None
):
    """
    Leo đồi khởi tạo ngẫu nhiên / lặp lại ngẫu nhiên.

    Ý tưởng:
    - Mỗi lần restart có 1 trạng thái bắt đầu.
    - Tại mỗi bước, xét các hàng xóm.
    - Chọn hàng xóm có h nhỏ nhất.
    - Nếu h hàng xóm tốt hơn h hiện tại thì đi tiếp.
    - Nếu không tốt hơn thì kẹt local minimum/plateau => restart.
    """
    if seed is not None:
        random.seed(seed)

    h_func = manhattan if heuristic == "manhattan" else misplaced_tiles
    labeler = Labeler()
    log_rows = []
    parent = {}
    global_reached = set()
    best_seen = None
    best_h = math.inf

    for restart in range(max_restarts + 1):
        if restart == 0 and use_given_start_first:
            current = start
        else:
            current = random_solvable_state(goal)

        parent[current] = (None, None)
        local_reached = {current}
        global_reached.add(current)
        labeler.get(current)

        for step in range(max_steps_per_restart + 1):
            cur_h = h_func(current, goal)

            if cur_h < best_h:
                best_h = cur_h
                best_seen = current

            if current == goal:
                log_rows.append({
                    "Lần": restart,
                    "Bước": step,
                    "Node": f"{labeler.get(current)}\n{state_to_text(current)}",
                    "h(Node)": cur_h,
                    "Beam/Current": labeler.get(current),
                    "Frontier / Neighbor đang xét": "Đã là Goal",
                    "Reached": len(global_reached),
                    "Chọn": "Dừng",
                    "Ghi chú": "Tìm thấy nghiệm"
                })
                return {
                    "found": True,
                    "goal_state": current,
                    "parent": parent,
                    "log": log_rows,
                    "best_state": best_seen,
                    "best_h": best_h,
                    "labeler": labeler,
                    "message": "Đã tìm thấy Goal bằng Random-Restart Hill Climbing."
                }

            neigh = []
            for action, nxt in neighbors(current):
                labeler.get(nxt)
                h = h_func(nxt, goal)
                neigh.append((h, action, nxt))

            neigh_sorted = sorted(neigh, key=lambda x: (x[0], MOVE_ORDER.index(x[1])))
            best_neighbor_h = neigh_sorted[0][0]
            best_candidates = [x for x in neigh_sorted if x[0] == best_neighbor_h]
            chosen_h, chosen_action, chosen_state = random.choice(best_candidates)

            frontier_text = " | ".join(
                f"({labeler.get(st)},{act},h={h})" for h, act, st in neigh_sorted
            )

            can_move = chosen_h < cur_h or (allow_sideways and chosen_h == cur_h and chosen_state not in local_reached)

            note = ""
            if can_move:
                parent.setdefault(chosen_state, (current, chosen_action))
                local_reached.add(chosen_state)
                global_reached.add(chosen_state)
                choose_text = f"{labeler.get(chosen_state)} qua {chosen_action}, h={chosen_h}"
                note = "Đi sang hàng xóm tốt hơn" if chosen_h < cur_h else "Đi ngang plateau"
            else:
                choose_text = "Không chọn"
                note = "Kẹt local minimum/plateau → restart"

            log_rows.append({
                "Lần": restart,
                "Bước": step,
                "Node": f"{labeler.get(current)}\n{state_to_text(current)}",
                "h(Node)": cur_h,
                "Beam/Current": labeler.get(current),
                "Frontier / Neighbor đang xét": frontier_text,
                "Reached": len(global_reached),
                "Chọn": choose_text,
                "Ghi chú": note
            })

            if can_move:
                current = chosen_state
            else:
                break

    return {
        "found": False,
        "goal_state": None,
        "parent": parent,
        "log": log_rows,
        "best_state": best_seen,
        "best_h": best_h,
        "labeler": labeler,
        "message": f"Chưa tìm thấy Goal. Trạng thái tốt nhất có h={best_h}."
    }

# ----------------------------
# 4. LOCAL BEAM SEARCH
# ----------------------------

def local_beam_search(
    start,
    goal,
    heuristic="manhattan",
    beam_width=4,
    max_iterations=100,
    use_given_start=True,
    seed=None
):
    """
    Local Beam Search.

    Ý tưởng:
    - Giữ k trạng thái tốt nhất cùng lúc.
    - Sinh tất cả hàng xóm từ k trạng thái hiện tại.
    - Chọn k hàng xóm có h nhỏ nhất làm beam mới.
    - Nếu một trạng thái là Goal thì dừng.
    """
    if seed is not None:
        random.seed(seed)

    h_func = manhattan if heuristic == "manhattan" else misplaced_tiles
    labeler = Labeler()
    parent = {}
    reached = set()
    log_rows = []

    beams = []
    if use_given_start:
        beams.append(start)
        parent[start] = (None, None)
        reached.add(start)

    while len(beams) < beam_width:
        st = random_solvable_state(goal)
        if st not in reached:
            beams.append(st)
            parent[st] = (None, None)
            reached.add(st)

    for st in beams:
        labeler.get(st)

    best_state = min(beams, key=lambda s: h_func(s, goal))
    best_h = h_func(best_state, goal)

    for it in range(max_iterations + 1):
        beams_sorted = sorted(beams, key=lambda s: h_func(s, goal))

        for st in beams_sorted:
            h = h_func(st, goal)
            if h < best_h:
                best_state, best_h = st, h

            if st == goal:
                log_rows.append({
                    "Lần": 0,
                    "Bước": it,
                    "Node": " / ".join(labeler.get(x) for x in beams_sorted),
                    "h(Node)": h,
                    "Beam/Current": " | ".join(f"{labeler.get(x)}(h={h_func(x, goal)})" for x in beams_sorted),
                    "Frontier / Neighbor đang xét": "Đã có Goal trong beam",
                    "Reached": len(reached),
                    "Chọn": labeler.get(st),
                    "Ghi chú": "Tìm thấy nghiệm"
                })
                return {
                    "found": True,
                    "goal_state": st,
                    "parent": parent,
                    "log": log_rows,
                    "best_state": best_state,
                    "best_h": best_h,
                    "labeler": labeler,
                    "message": "Đã tìm thấy Goal bằng Local Beam Search."
                }

        candidates = []
        seen_candidate = set()

        for beam_state in beams_sorted:
            for action, nxt in neighbors(beam_state):
                labeler.get(nxt)
                h = h_func(nxt, goal)

                # Tránh lặp lại trạng thái đã reached để giảm vòng lặp
                if nxt in reached:
                    continue

                if nxt not in seen_candidate:
                    seen_candidate.add(nxt)
                    candidates.append((h, action, beam_state, nxt))
                    parent.setdefault(nxt, (beam_state, action))

        candidates_sorted = sorted(
            candidates,
            key=lambda x: (x[0], labeler.get(x[2]), MOVE_ORDER.index(x[1]))
        )

        frontier_text = " | ".join(
            f"({labeler.get(parent_st)} --{act}--> {labeler.get(nxt)}, h={h})"
            for h, act, parent_st, nxt in candidates_sorted[:20]
        )
        if len(candidates_sorted) > 20:
            frontier_text += f" | ... còn {len(candidates_sorted) - 20} ứng viên"

        if not candidates_sorted:
            log_rows.append({
                "Lần": 0,
                "Bước": it,
                "Node": " / ".join(labeler.get(x) for x in beams_sorted),
                "h(Node)": best_h,
                "Beam/Current": " | ".join(f"{labeler.get(x)}(h={h_func(x, goal)})" for x in beams_sorted),
                "Frontier / Neighbor đang xét": "Rỗng",
                "Reached": len(reached),
                "Chọn": "Không chọn",
                "Ghi chú": "Không còn ứng viên mới → dừng"
            })
            break

        chosen = candidates_sorted[:beam_width]
        new_beams = [nxt for h, act, parent_st, nxt in chosen]

        for st in new_beams:
            reached.add(st)

        chosen_text = " | ".join(
            f"{labeler.get(nxt)}(h={h})"
            for h, act, parent_st, nxt in chosen
        )

        log_rows.append({
            "Lần": 0,
            "Bước": it,
            "Node": " / ".join(labeler.get(x) for x in beams_sorted),
            "h(Node)": min(h_func(x, goal) for x in beams_sorted),
            "Beam/Current": " | ".join(f"{labeler.get(x)}(h={h_func(x, goal)})" for x in beams_sorted),
            "Frontier / Neighbor đang xét": frontier_text,
            "Reached": len(reached),
            "Chọn": chosen_text,
            "Ghi chú": f"Giữ lại {beam_width} trạng thái tốt nhất"
        })

        beams = new_beams

    return {
        "found": False,
        "goal_state": None,
        "parent": parent,
        "log": log_rows,
        "best_state": best_state,
        "best_h": best_h,
        "labeler": labeler,
        "message": f"Chưa tìm thấy Goal. Trạng thái tốt nhất có h={best_h}."
    }


# ----------------------------
# 5. SIMULATED ANNEALING
# ----------------------------

def simulated_annealing_search(
    start,
    goal,
    heuristic="manhattan",
    max_iterations=500,
    initial_temperature=10.0,
    cooling_rate=0.97,
    min_temperature=0.001,
    seed=None
):
    """
    Simulated Annealing.

    Ý tưởng:
    - Vẫn ưu tiên đi sang trạng thái tốt hơn.
    - Đôi khi chấp nhận trạng thái xấu hơn với xác suất exp(-Δh/T).
    - T giảm dần qua mỗi vòng lặp, càng về sau càng ít nhận bước xấu.
    """
    if seed is not None:
        random.seed(seed)

    h_func = manhattan if heuristic == "manhattan" else misplaced_tiles
    labeler = Labeler()
    parent = {start: (None, None)}
    reached = {start}
    log_rows = []
    current = start
    labeler.get(current)
    best_seen = current
    best_h = h_func(current, goal)
    temperature = float(initial_temperature)
    actual_path = [(start, None)]

    for it in range(max_iterations + 1):
        cur_h = h_func(current, goal)
        if cur_h < best_h:
            best_seen, best_h = current, cur_h

        if current == goal:
            log_rows.append({
                "Lần": 0,
                "Bước": it,
                "Node": f"{labeler.get(current)}\n{state_to_text(current)}",
                "h(Node)": cur_h,
                "Beam/Current": f"{labeler.get(current)} | T={temperature:.4f}",
                "Frontier / Neighbor đang xét": "Đã là Goal",
                "Reached": len(reached),
                "Chọn": "Dừng",
                "Ghi chú": "Tìm thấy nghiệm"
            })
            return {
                "found": True,
                "goal_state": current,
                "parent": parent,
                "path": actual_path,
                "log": log_rows,
                "best_state": best_seen,
                "best_h": best_h,
                "labeler": labeler,
                "message": "Đã tìm thấy Goal bằng Simulated Annealing."
            }

        if temperature < min_temperature:
            log_rows.append({
                "Lần": 0,
                "Bước": it,
                "Node": f"{labeler.get(current)}\n{state_to_text(current)}",
                "h(Node)": cur_h,
                "Beam/Current": f"{labeler.get(current)} | T={temperature:.4f}",
                "Frontier / Neighbor đang xét": "Nhiệt độ quá thấp",
                "Reached": len(reached),
                "Chọn": "Không chọn",
                "Ghi chú": "Dừng do T < Tmin"
            })
            break

        neigh = []
        for action, nxt in neighbors(current):
            labeler.get(nxt)
            h = h_func(nxt, goal)
            neigh.append((h, action, nxt))

        chosen_h, chosen_action, chosen_state = random.choice(neigh)
        delta = chosen_h - cur_h
        if delta <= 0:
            prob = 1.0
            accepted = True
            reason = "Tốt hơn hoặc bằng nên nhận"
        else:
            prob = math.exp(-delta / temperature)
            r = random.random()
            accepted = r < prob
            reason = f"Xấu hơn Δh={delta}, p={prob:.4f}, random={r:.4f}"

        frontier_text = " | ".join(f"({labeler.get(st)},{act},h={h})" for h, act, st in sorted(neigh, key=lambda x: (x[0], MOVE_ORDER.index(x[1]))))
        choose_text = f"{labeler.get(chosen_state)} qua {chosen_action}, h={chosen_h}" if accepted else "Không nhận bước ngẫu nhiên"
        note = reason + (" → đi tiếp" if accepted else " → giữ nguyên")

        log_rows.append({
            "Lần": 0,
            "Bước": it,
            "Node": f"{labeler.get(current)}\n{state_to_text(current)}",
            "h(Node)": cur_h,
            "Beam/Current": f"{labeler.get(current)} | T={temperature:.4f}",
            "Frontier / Neighbor đang xét": frontier_text,
            "Reached": len(reached),
            "Chọn": choose_text,
            "Ghi chú": note
        })

        if accepted:
            if chosen_state not in parent:
                parent[chosen_state] = (current, chosen_action)
            reached.add(chosen_state)
            current = chosen_state
            actual_path.append((current, chosen_action))

        temperature *= cooling_rate

    return {
        "found": False,
        "goal_state": None,
        "parent": parent,
        "path": actual_path,
        "log": log_rows,
        "best_state": best_seen,
        "best_h": best_h,
        "labeler": labeler,
        "message": f"Chưa tìm thấy Goal. Simulated Annealing dừng, trạng thái tốt nhất có h={best_h}."
    }

# ----------------------------
# 6. GIAO DIỆN JUPYTER
# ----------------------------

def html_card(title_text, body_widget, badge=None):
    badge_html = f'<span class="puzzle-pill">{badge}</span>' if badge else ''
    header = widgets.HTML(f'<div class="puzzle-card-title"><h3>{title_text}</h3>{badge_html}</div>')
    return widgets.VBox([header, body_widget], layout=widgets.Layout(
        border="1px solid #e2e8f0", padding="14px", border_radius="22px",
        margin="0 0 12px 0", box_shadow="0 10px 28px rgba(15,23,42,.08)"
    ))

def section_html(text):
    return widgets.HTML(f'<div style="font-weight:800;color:#0f172a;margin:10px 0 6px;">{text}</div>')

title = widgets.HTML(APP_CSS + '''
<div class="puzzle-app"><div class="puzzle-hero">
<h2>8-Puzzle Local Search</h2>
<p>Giao diện mô phỏng <b>Hill Climbing</b>, <b>Random-Restart</b>, <b>Local Beam Search</b> và <b>Simulated Annealing</b>. Chọn thuật toán bằng nút, sau đó chạy để xem đường đi, node, frontier và reached.</p>
</div></div>
''')

COMMON_TEXTAREA_LAYOUT = widgets.Layout(width="100%", height="112px")
COMMON_CONTROL_LAYOUT = widgets.Layout(width="100%")

start_input = widgets.Textarea(value="2 8 3\n1 6 4\n7 0 5", description="", placeholder="Ví dụ:\n2 8 3\n1 6 4\n7 0 5", layout=COMMON_TEXTAREA_LAYOUT)
goal_input = widgets.Textarea(value="1 2 3\n8 0 4\n7 6 5", description="", placeholder="Ví dụ:\n1 2 3\n8 0 4\n7 6 5", layout=COMMON_TEXTAREA_LAYOUT)

algo_buttons = widgets.ToggleButtons(
    options=[
        ("Hill Climbing", "hc"),
        ("Random-Restart", "rrhc"),
        ("Local Beam", "beam"),
        ("Simulated Annealing", "sa"),
    ],
    value="hc",
    description="Thuật toán:",
    button_style="",
    tooltips=[
        "Hill-climbing search",
        "Random-restart hill-climbing search",
        "Local beam search",
        "Simulated annealing",
    ],
    layout=widgets.Layout(width="100%")
)
algo_box = widgets.Box([algo_buttons], layout=widgets.Layout(width="100%"))
algo_box.add_class("algo-switch")
heuristic_dropdown = widgets.Dropdown(options=[("Manhattan", "manhattan"), ("Số ô sai vị trí", "misplaced")], value="manhattan", description="h(n):", layout=COMMON_CONTROL_LAYOUT)
restart_slider = widgets.IntSlider(value=30, min=0, max=200, step=1, description="Restart:", continuous_update=False, layout=COMMON_CONTROL_LAYOUT)
step_slider = widgets.IntSlider(value=60, min=1, max=300, step=1, description="Step:", continuous_update=False, layout=COMMON_CONTROL_LAYOUT)
sideways_checkbox = widgets.Checkbox(value=False, description="Cho phép đi ngang khi h bằng nhau", indent=False, layout=COMMON_CONTROL_LAYOUT)
beam_width_slider = widgets.IntSlider(value=5, min=2, max=20, step=1, description="Beam k:", continuous_update=False, layout=COMMON_CONTROL_LAYOUT)
iteration_slider = widgets.IntSlider(value=100, min=1, max=500, step=1, description="Lặp:", continuous_update=False, layout=COMMON_CONTROL_LAYOUT)
hc_step_slider = widgets.IntSlider(value=100, min=1, max=500, step=1, description="Step HC:", continuous_update=False, layout=COMMON_CONTROL_LAYOUT)
sa_iteration_slider = widgets.IntSlider(value=500, min=10, max=5000, step=10, description="Lặp SA:", continuous_update=False, layout=COMMON_CONTROL_LAYOUT)
sa_temp_slider = widgets.FloatSlider(value=10.0, min=0.1, max=100.0, step=0.1, description="T ban đầu:", continuous_update=False, layout=COMMON_CONTROL_LAYOUT)
sa_cooling_slider = widgets.FloatSlider(value=0.97, min=0.80, max=0.999, step=0.001, description="Cooling:", readout_format=".3f", continuous_update=False, layout=COMMON_CONTROL_LAYOUT)
use_start_checkbox = widgets.Checkbox(value=True, description="Dùng Start người nhập làm trạng thái đầu tiên", indent=False, layout=COMMON_CONTROL_LAYOUT)
seed_input = widgets.IntText(value=42, description="Seed:", layout=widgets.Layout(width="170px"))

run_button = widgets.Button(description="Chạy thuật toán", button_style="primary", icon="play", layout=widgets.Layout(width="180px", height="42px"))
random_button = widgets.Button(description="Random Start", button_style="info", icon="random", layout=widgets.Layout(width="160px", height="40px"))
clear_button = widgets.Button(description="Xóa kết quả", button_style="warning", icon="trash", layout=widgets.Layout(width="135px", height="40px"))
preview_button = widgets.Button(description="Xem Start/Goal", button_style="", icon="eye", layout=widgets.Layout(width="155px", height="40px"))
output = widgets.Output(layout=widgets.Layout(width="100%"))

def set_algorithm_visibility(*_):
    algo = algo_buttons.value
    hc_display = None if algo == "hc" else "none"
    rrhc_display = None if algo == "rrhc" else "none"
    beam_display = None if algo == "beam" else "none"
    sa_display = None if algo == "sa" else "none"
    hc_step_slider.layout.display = hc_display
    sideways_checkbox.layout.display = None if algo in ("hc", "rrhc") else "none"
    restart_slider.layout.display = rrhc_display
    step_slider.layout.display = rrhc_display
    beam_width_slider.layout.display = beam_display
    iteration_slider.layout.display = beam_display
    sa_iteration_slider.layout.display = sa_display
    sa_temp_slider.layout.display = sa_display
    sa_cooling_slider.layout.display = sa_display

algo_buttons.observe(set_algorithm_visibility, names="value")
set_algorithm_visibility()

def preview_start_goal(message="Nhấn “Chạy thuật toán” để bắt đầu."):
    with output:
        clear_output()
        try:
            st = parse_state(start_input.value)
            gl = parse_state(goal_input.value)
            ok = is_solvable(st, gl)
            solvable_msg = "Có thể giải" if ok else "Không thể giải"
            alert_class = "puzzle-alert-good" if ok else "puzzle-alert-bad"
            display(HTML(f'<div class="puzzle-note {alert_class}"><b>{message}</b><br>Trạng thái hiện tại: {solvable_msg}. Manhattan h={manhattan(st, gl)}.</div>'))
            show_boards([st, gl], gl, ["Start", "Goal"], [f"h={manhattan(st, gl)}", "Đích"])
        except Exception as e:
            display(HTML(f'<div class="puzzle-note puzzle-alert-bad"><b>Lỗi nhập ma trận:</b> {e}</div>'))

def randomize_start(_=None):
    try:
        goal = parse_state(goal_input.value)
        st = random_solvable_state(goal)
        start_input.value = state_to_text(st)
        preview_start_goal("Đã tạo Start ngẫu nhiên hợp lệ.")
    except Exception as e:
        with output:
            clear_output()
            display(HTML(f'<div class="puzzle-note puzzle-alert-bad"><b>Lỗi:</b> {e}</div>'))

def clear_result(_=None):
    with output:
        clear_output()
        display(HTML('<div class="puzzle-note">Đã xóa kết quả. Bạn có thể nhập lại Start/Goal hoặc chạy thuật toán.</div>'))

def node_to_html(value):
    """Hiển thị cột Node thành nhãn + bảng 3x3 nhỏ để dễ nhìn hơn."""
    text = str(value).replace("\\n", "\n")
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if not lines:
        return ""
    label = lines[0]
    nums = []
    for line in lines[1:]:
        nums.extend(re.findall(r"-?\d+", line))
    if len(nums) != 9:
        return text.replace("\n", "<br>")
    cells = "".join(
        f'<div class="puzzle-mini-cell {"zero" if x == "0" else ""}">{"" if x == "0" else x}</div>'
        for x in nums
    )
    return f'<div class="puzzle-mini-node"><div class="puzzle-mini-label">{label}</div><div class="puzzle-mini-grid">{cells}</div></div>'

def htmlize_log_text(value):
    text = str(value).replace("\\n", "\n")
    return text.replace("\n", "<br>").replace(" | ", "<br>")

def styled_dataframe(df, max_rows=150):
    small = df.head(max_rows).copy()
    if "Node" in small.columns:
        small["Node"] = small["Node"].apply(node_to_html)
    for col in ["Frontier / Neighbor đang xét", "Chọn", "Ghi chú", "Beam/Current"]:
        if col in small.columns:
            small[col] = small[col].apply(htmlize_log_text)
    html = small.to_html(index=False, escape=False)
    return HTML(f'<div class="puzzle-table-wrap">{html}</div>')

def run_algorithm(_=None):
    with output:
        clear_output()
        try:
            start = parse_state(start_input.value)
            goal = parse_state(goal_input.value)
        except Exception as e:
            display(HTML(f'<div class="puzzle-note puzzle-alert-bad"><b>Lỗi nhập ma trận:</b> {e}</div>'))
            return
        if not is_solvable(start, goal):
            display(HTML('<div class="puzzle-note puzzle-alert-bad"><b>Start không thể biến đổi về Goal.</b><br>Với 8-puzzle 3x3, parity số nghịch thế của Start và Goal không phù hợp.</div>'))
            show_boards([start, goal], goal, ["Start", "Goal"], ["Không solvable", "Đích"])
            return
        heuristic_name = "Manhattan" if heuristic_dropdown.value == "manhattan" else "Số ô sai vị trí"
        algorithm_names = {
            "hc": "Hill Climbing Search",
            "rrhc": "Random-Restart Hill Climbing",
            "beam": "Local Beam Search",
            "sa": "Simulated Annealing",
        }
        algorithm_name = algorithm_names[algo_buttons.value]
        display(HTML(f'<div class="puzzle-note"><b>Cấu hình chạy</b><br>Thuật toán: <b>{algorithm_name}</b> · Heuristic: <b>{heuristic_name}</b> · Seed: <b>{seed_input.value}</b></div>'))
        show_boards([start, goal], goal, ["Start", "Goal"], [f"h={manhattan(start, goal)} theo Manhattan", "Đích cần đạt"])
        if algo_buttons.value == "hc":
            result = hill_climbing_search(start=start, goal=goal, heuristic=heuristic_dropdown.value, max_steps=hc_step_slider.value, allow_sideways=sideways_checkbox.value, seed=seed_input.value)
        elif algo_buttons.value == "rrhc":
            result = random_restart_hill_climbing(start=start, goal=goal, heuristic=heuristic_dropdown.value, max_restarts=restart_slider.value, max_steps_per_restart=step_slider.value, use_given_start_first=use_start_checkbox.value, allow_sideways=sideways_checkbox.value, seed=seed_input.value)
        elif algo_buttons.value == "beam":
            result = local_beam_search(start=start, goal=goal, heuristic=heuristic_dropdown.value, beam_width=beam_width_slider.value, max_iterations=iteration_slider.value, use_given_start=use_start_checkbox.value, seed=seed_input.value)
        else:
            result = simulated_annealing_search(start=start, goal=goal, heuristic=heuristic_dropdown.value, max_iterations=sa_iteration_slider.value, initial_temperature=sa_temp_slider.value, cooling_rate=sa_cooling_slider.value, seed=seed_input.value)
        alert_class = "puzzle-alert-good" if result["found"] else "puzzle-alert-warn"
        display(HTML(f'<div class="puzzle-note {alert_class}" style="margin-top:12px;"><b>Kết quả:</b> {result["message"]}</div>'))
        if result["found"]:
            path = result.get("path") or path_from_parent(result["parent"], result["goal_state"])
            states = [st for st, act in path]
            titles, subtitles = [], []
            for i, (st, act) in enumerate(path):
                if i == 0:
                    titles.append("Bắt đầu"); subtitles.append(f"h={manhattan(st, goal)}")
                else:
                    titles.append(f"Bước {i}"); subtitles.append(f"Đi {act}, h={manhattan(st, goal)}")
            display(HTML(f'<h3 class="puzzle-section-title">Đường đi nghiệm: {len(states)-1} bước</h3>'))
            show_boards(states, goal, titles, subtitles, max_boards=30)
            if len(states) > 30:
                display(HTML(f'<div class="puzzle-note">Đường đi dài {len(states)} trạng thái, chỉ hiển thị 30 trạng thái đầu.</div>'))
        else:
            best = result["best_state"]
            display(HTML('<h3 class="puzzle-section-title">Trạng thái tốt nhất thuật toán tìm được</h3>'))
            show_boards([best], goal, ["Best seen"], [f"h={result['best_h']}"])
        df = make_log_dataframe(result["log"])
        display(HTML('<h3 class="puzzle-section-title">Bảng Node / Frontier / Reached</h3>'))
        if len(df) == 0:
            display(HTML('<div class="puzzle-note">Không có log.</div>'))
        else:
            display(styled_dataframe(df, max_rows=150))
            if len(df) > 150:
                display(HTML(f'<div class="puzzle-note">Bảng có {len(df)} dòng, chỉ hiển thị 150 dòng đầu.</div>'))
        display(HTML('<div class="puzzle-note" style="margin-top:12px;"><b>Ghi nhớ:</b><br>- Hill Climbing chỉ nhìn hàng xóm hiện tại nên dễ kẹt local minimum/plateau/ridge.<br>- Random-Restart khắc phục bằng cách thử nhiều điểm bắt đầu ngẫu nhiên.<br>- Local Beam Search giữ k trạng thái tốt nhất cùng lúc nên rộng hơn Hill Climbing.<br>- Simulated Annealing đôi khi nhận bước xấu hơn để thoát kẹt, nhưng vẫn không đảm bảo luôn tìm ra nghiệm.</div>'))

random_button.on_click(randomize_start)
clear_button.on_click(clear_result)
preview_button.on_click(lambda _: preview_start_goal())
run_button.on_click(run_algorithm)

input_card = html_card("Nhập trạng thái", widgets.VBox([
    widgets.HTML('<div style="font-weight:700;margin-bottom:5px;color:#334155;">Start</div>'), start_input,
    widgets.HTML('<div style="font-weight:700;margin:10px 0 5px;color:#334155;">Goal</div>'), goal_input,
    widgets.HBox([random_button, preview_button, clear_button], layout=widgets.Layout(gap="8px", flex_flow="row wrap")),
    widgets.HTML('<div style="font-size:12px;color:#64748b;margin-top:8px;">Có thể nhập dạng 3 dòng hoặc 1 dòng có đủ 9 số từ 0 đến 8.</div>')
]), badge="Input")

setting_card = html_card("Cài đặt thuật toán", widgets.VBox([
    algo_box, heuristic_dropdown,
    section_html("Tham số Hill Climbing"), hc_step_slider, sideways_checkbox,
    section_html("Tham số Random-Restart Hill Climbing"), restart_slider, step_slider,
    section_html("Tham số Local Beam Search"), beam_width_slider, iteration_slider,
    section_html("Tham số Simulated Annealing"), sa_iteration_slider, sa_temp_slider, sa_cooling_slider,
    section_html("Tùy chọn chung"), use_start_checkbox,
    widgets.HBox([seed_input, run_button], layout=widgets.Layout(gap="12px", align_items="center", flex_flow="row wrap"))
], layout=widgets.Layout(gap="6px")), badge="Config")


issues_card = html_card("Issues of Hill-Climbing Search", widgets.HTML('''
<div style="line-height:1.55;color:#334155;font-size:13px;">
    <b>Local minimum:</b> bị kẹt ở trạng thái tốt hơn xung quanh nhưng chưa phải Goal.<br>
    <b>Plateau:</b> nhiều trạng thái có cùng h nên khó biết đi hướng nào.<br>
    <b>Ridge/Shoulder:</b> cần đi tạm sang ngang hoặc xấu hơn mới tới vùng tốt hơn.<br>
    <b>Cách khắc phục:</b> Random-Restart, cho phép sideways move, Local Beam Search hoặc Simulated Annealing.
</div>
'''), badge="Theory")
app_layout = widgets.VBox([title, widgets.HBox([input_card, setting_card], layout=widgets.Layout(gap="16px", align_items="flex-start", flex_flow="row wrap")), issues_card, output], layout=widgets.Layout(width="100%"))
display(app_layout)
preview_start_goal("Sẵn sàng. Kiểm tra Start/Goal ban đầu bên dưới.")
